# Raw Data Download

Download raw experimental data files from the aimdl/datafiles API with configurable data types.

This notebook demonstrates how to:
- Query files by data type using the aimdl/datafiles endpoint
- Download multiple files in parallel
- Save to disk with metadata

In [4]:
import sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from aimdl_examples import get_girder_client, download_item_to_disk

/Users/alirachidi/miniconda3/lib/python3.9/site-packages/girder_client/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


## Configuration

In [5]:
from dotenv import load_dotenv

load_dotenv('..')

# Configure data type to download
# OPTIONS: pdv_trace OR pdv_alpss_output 
DATA_TYPE = "pdv_alpss_output"

regex_expression = "JHAMAB00021-16"

# Output directory for downloaded files
OUTPUT_DIR = f"./raw_data_{DATA_TYPE}"

## Download Files

In [6]:
with requests.Session() as session:
    gc = get_girder_client(session=session)
    results = []
    limit = 50
    offset = 0
    
    with ThreadPoolExecutor(max_workers=8) as executor:
        while True:
            batch = gc.get(
                "aimdl/datafiles",
                parameters={
                    "limit": limit,
                    "offset": offset,
                    "dataType": DATA_TYPE,
                },
            )
            if not batch:
                break
            
            futures = [
                executor.submit(download_item_to_disk, gc, item, OUTPUT_DIR)
                for item in batch
            ]
            
            for future in as_completed(futures):
                try:
                    result = future.result()
                    results.append(result)
                    print(f"Downloaded: {result['name']} ({result['size'] / 1024:.2f} KB)")
                except Exception as e:
                    print(f"Error: {e}")
            
            if len(batch) < limit:
                break
            offset += limit

print(f"\\nTotal files downloaded: {len(results)}")
print(f"Total size: {sum(r['size'] for r in results) / (1024**2):.2f} MB")

Downloaded: _2026-05-07_21-22-22_shot01_ch1-inputs.csv (1.20 KB)
Downloaded: _2026-05-07_21-22-22_shot01_ch1-iq.png (66.71 KB)
Downloaded: _2026-05-07_21-22-22_shot01_ch1-velocity--smooth.csv (958.29 KB)
Downloaded: _2026-05-07_21-22-39_shot02_ch1-inputs.csv (1.20 KB)
Downloaded: _2026-05-07_21-22-39_shot02_ch1-hel.png (144.46 KB)
Downloaded: _2026-05-07_21-22-22_shot01_ch1-hel.png (142.33 KB)
Downloaded: _2026-05-07_21-22-22_shot01_ch1-plots.png (586.38 KB)
Downloaded: _2026-05-07_21-22-39_shot02_ch1-iq.png (69.40 KB)
Downloaded: _2026-05-07_21-22-39_shot02_ch1-velocity.csv (956.25 KB)
Downloaded: _2026-05-07_21-22-39_shot02_ch1-velocity--smooth.csv (956.25 KB)
Downloaded: _2026-05-07_21-22-22_shot01_ch1-noisefrac.csv (939.50 KB)
Downloaded: _2026-05-07_21-22-39_shot02_ch1-noisefrac.csv (937.50 KB)
Downloaded: _2026-05-07_21-22-50_shot03_ch1-hel.png (152.36 KB)
Downloaded: _2026-05-07_21-22-22_shot01_ch1-veluncert.csv (939.50 KB)
Downloaded: _2026-05-07_21-22-50_shot03_ch1-inputs.csv 

KeyboardInterrupt: 

## Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print(f"Downloaded {len(df)} files of type '{DATA_TYPE}'")
print(f"Output directory: {OUTPUT_DIR}")
print(f"\\nSample files:")
print(df[["name", "size"]].head(10))